In [1]:
import os, json
import numpy as np

meta_file = 'metadata - old DeepSTABp-lysates dataset (train & valid set), available from dp180.csv'
accession_file = 'accessions - cell_lysate-0_1 - 20241121 - list of files inferred from deepstabp code.txt'

dst_file = 'the fold.json'

In [2]:
metadata_raw = np.loadtxt(
    meta_file,
    delimiter=',',
    dtype=np.str_
)
assert metadata_raw.shape[0] == np.unique(metadata_raw[:, 0]).size

accessions_of_interest = np.loadtxt(
    accession_file,
    delimiter=',',
    dtype=np.str_
)
assert accessions_of_interest.size == np.unique(accessions_of_interest).size
print('# accessions of interest:', accessions_of_interest.size)

# find all accessions available in metadata_raw
accessions = np.array([
    acc for acc in accessions_of_interest
    if acc in metadata_raw[:, 0]
])
print('# accessions of interst present in metadata:',
      accessions.size)

metadata = np.concatenate(
    [
        metadata_raw[metadata_raw[:, 0] == acc]
        for acc in accessions
    ]
)

# print(metadata.shape)

# accessions of interest: 19135
# accessions of interst present in metadata: 11899


In [3]:
# randomize
accessions = np.char.add(np.random.permutation(accessions), '-AFv4')

n_folds = 10
n_entries_per_fold = 5

folds = [
    accessions[i:i+n_entries_per_fold].tolist()
    for i in range(0, n_folds*n_entries_per_fold, n_entries_per_fold)
]
print(len(folds))

10


In [4]:
mini_fold = {
    'train': {},
    'valid': {}
}

for fold_idx in range(n_folds):

    # omit the fold_idx-th fold
    train_set = [
        acc
        for i, fold_entries in enumerate(folds)
        for acc in fold_entries
        if i != fold_idx
    ]
    valid_set = folds[fold_idx]

    mini_fold['train'][fold_idx] = train_set
    mini_fold['valid'][fold_idx] = valid_set

In [5]:
with open(dst_file, 'w') as f:
    json.dump(mini_fold, f, indent=2)

np.savetxt(
    'metadata - sandbox_50 (example).csv',
    np.concatenate([
        metadata[metadata[:, 0] == acc[:-5]]
        for acc in np.sort(np.concatenate(folds))
    ]),
    delimiter=',',
    fmt='%s'
)

In [6]:
# assert that the training set and test set are disjoint
train_set = np.concatenate(folds)
test_set = np.loadtxt(
    '../metadata - old DeepSTABp-lysates dataset (test set), available from dp180.csv',
    delimiter=',',
    dtype=np.str_
)
assert np.intersect1d(train_set, test_set[:, 0]).size == 0

test_set = np.loadtxt(
    '../metadata - DeepTM benchmark set 2, available from dp180.csv',
    delimiter=',',
    dtype=np.str_
)
assert np.intersect1d(train_set, test_set[:, 0]).size == 0